# 🔮 Candle Forecast vs Reality

**Workflow:**
1. Pick symbol, timeframe, and a historical date/time
2. Fetch candles up to that point → run Kronos TA + LLM forecast
3. Fetch the *next* N real candles that followed
4. Compare forecast direction / levels against reality
5. Score: hit / miss / partial

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
SYMBOL       = "BTCUSDT"          # Delta: BTCUSDT | NSE: RELIANCE
SOURCE       = "delta"            # delta | fyers
INTERVAL     = "5m"               # 1m 5m 15m 1h 4h 1d
FORECAST_AT  = "2025-06-20 10:00" # snapshot time (IST) — forecast from here
FORECAST_N   = 6                  # how many candles ahead to forecast
LLM_PROVIDER = "openai"           # openai | ollama

In [ ]:
import asyncio, json, sys, os
from datetime import datetime, timezone, timedelta
from pathlib import Path

# Add project root to path
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

print("Project root:", ROOT)

In [ ]:
# ── Fetch candles via Delta/Fyers API directly ────────────────────────────────
from datetime import datetime, timezone, timedelta

def parse_ist(dt_str: str) -> datetime:
    """Parse 'YYYY-MM-DD HH:MM' as IST (UTC+5:30) and return UTC datetime."""
    ist_offset = timedelta(hours=5, minutes=30)
    naive = datetime.strptime(dt_str, "%Y-%m-%d %H:%M")
    return (naive - ist_offset).replace(tzinfo=timezone.utc)

snapshot_utc = parse_ist(FORECAST_AT)
print(f"Snapshot UTC: {snapshot_utc}")

# Lookback: 7 days of history ending at snapshot
lookback_from = snapshot_utc - timedelta(days=7)

In [ ]:
# ── Fetch historical candles (up to snapshot) ─────────────────────────────────
def fetch_candles_range(symbol, source, interval, from_dt, to_dt):
    """Synchronous fetch — Fyers uses requests (not async), Delta uses httpx async.
    We wrap Delta in asyncio.run to keep the interface uniform."""
    if source == "delta":
        import asyncio as _asyncio
        from backend.brokers.delta.source import DeltaSource
        from backend.config import settings
        src = DeltaSource(
            api_key=settings.DELTA_API_KEY,
            api_secret=settings.DELTA_API_SECRET,
            region=settings.DELTA_REGION,
        )
        async def _fetch():
            try:
                return await src.fetch_ohlc(
                    symbol=symbol, from_dt=from_dt, to_dt=to_dt, interval=interval
                )
            finally:
                await src.close()
        # In a Jupyter notebook the event loop is already running — use nest_asyncio
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except ImportError:
            pass
        return _asyncio.get_event_loop().run_until_complete(_fetch())

    else:  # fyers — synchronous
        from backend.brokers.fyers.source import FyersSource
        from backend.config import settings
        src = FyersSource(
            client_id=settings.FYERS_CLIENT_ID,
            access_token=settings.FYERS_ACCESS_TOKEN,
        )
        return src.fetch_ohlc(
            symbol=symbol, from_dt=from_dt, to_dt=to_dt, interval=interval
        )


hist_df = fetch_candles_range(SYMBOL, SOURCE, INTERVAL, lookback_from, snapshot_utc)
print(f"Historical candles fetched: {len(hist_df)}")
hist_df.tail()

In [ ]:
# ── Convert to Kronos candle format ───────────────────────────────────────────
def df_to_candles(df):
    rows = df.reset_index()
    result = []
    for _, row in rows.iterrows():
        result.append({
            "datetime": str(row.get("datetime", row.name)),
            "open":   float(row["open"]),
            "high":   float(row["high"]),
            "low":    float(row["low"]),
            "close":  float(row["close"]),
            "volume": float(row["volume"]),
        })
    return result

hist_candles = df_to_candles(hist_df)
print(f"Last 3 historical candles:")
for c in hist_candles[-3:]:
    print(f"  {c['datetime']}  O={c['open']} H={c['high']} L={c['low']} C={c['close']}")

In [ ]:
# ── Run Kronos TA on historical candles ───────────────────────────────────────
from backend.agents.kronos_agent import KronosAgent
from backend.protocol import Task

kronos = KronosAgent()
kr_task = Task(
    task_id="forecast-kronos",
    agent="kronos_agent",
    input={"symbol": SYMBOL, "source": SOURCE, "interval": INTERVAL, "candles": hist_candles},
)
kr_resp = await kronos.handle_task(kr_task)
kronos_ctx = kr_resp.artifacts[0].data if kr_resp.artifacts else {}

print("=== KRONOS CONTEXT ===")
print(f"Trend:    {kronos_ctx.get('trend')}")
print(f"RSI:      {kronos_ctx.get('rsi')}  ({kronos_ctx.get('signal',{}).get('rsi_zone')})")
print(f"EMA9/21/50: {kronos_ctx.get('ema9')} / {kronos_ctx.get('ema21')} / {kronos_ctx.get('ema50')}")
print(f"Patterns: {kronos_ctx.get('patterns')}")
print(f"Bias:     {kronos_ctx.get('signal',{}).get('bias')} (score={kronos_ctx.get('signal',{}).get('score')})")
print(f"Support:  {kronos_ctx.get('support')}")
print(f"Resistance: {kronos_ctx.get('resistance')}")
print(f"CPR:      {kronos_ctx.get('cpr')}")

In [ ]:
# ── LLM Forecast ─────────────────────────────────────────────────────────────
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

FORECAST_SYSTEM = f"""You are a professional quantitative analyst.
You are given historical OHLCV candles and a full technical analysis context for {SYMBOL}.
Based on this data, forecast the next {FORECAST_N} candles.

Respond ONLY with valid JSON in this format:
{{
  "direction": "bullish|bearish|sideways",
  "confidence": <0-1 float>,
  "rationale": "<1-2 sentence technical reasoning>",
  "forecast_candles": [
    {{
      "candle": 1,
      "open": <float>,
      "high": <float>,
      "low": <float>,
      "close": <float>,
      "note": "<optional pattern or level note>"
    }}
    // ... repeat for all {FORECAST_N} candles
  ],
  "key_levels": {{
    "target": <float>,
    "stop_loss": <float>
  }}
}}"""

sig = kronos_ctx.get("signal", {})
recent_20 = hist_candles[-20:]
user_prompt = f"""
Symbol: {SYMBOL}  Interval: {INTERVAL}  Snapshot: {FORECAST_AT} IST

=== KRONOS TECHNICAL CONTEXT ===
Trend: {kronos_ctx.get('trend','?').upper()}
EMA9={kronos_ctx.get('ema9')}  EMA21={kronos_ctx.get('ema21')}  EMA50={kronos_ctx.get('ema50')}
RSI(14): {kronos_ctx.get('rsi')} ({sig.get('rsi_zone','?')})
ATR: {kronos_ctx.get('atr')}
Support: {kronos_ctx.get('support')}
Resistance: {kronos_ctx.get('resistance')}
CPR: {json.dumps(kronos_ctx.get('cpr', {}))}
Patterns: {kronos_ctx.get('patterns', [])}
Bias: {sig.get('bias','?')} (score={sig.get('score',0)})
================================

Last {len(recent_20)} candles (most recent last):
{json.dumps(recent_20, indent=2)}

Forecast the next {FORECAST_N} {INTERVAL} candles from {FORECAST_AT} IST.
"""

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, api_key=os.getenv("OPENAI_API_KEY"))
response = await llm.ainvoke([
    SystemMessage(content=FORECAST_SYSTEM),
    HumanMessage(content=user_prompt),
])

import re
raw = re.sub(r"```json|```", "", response.content).strip()
forecast = json.loads(raw)

print("=== LLM FORECAST ===")
print(f"Direction:  {forecast.get('direction').upper()}")
print(f"Confidence: {forecast.get('confidence'):.0%}")
print(f"Rationale:  {forecast.get('rationale')}")
print(f"Target:     {forecast.get('key_levels',{}).get('target')}")
print(f"Stop Loss:  {forecast.get('key_levels',{}).get('stop_loss')}")

In [ ]:
# ── Fetch REAL candles that followed ─────────────────────────────────────────
INTERVAL_MINS = {"1m":1,"3m":3,"5m":5,"15m":15,"30m":30,"1h":60,"4h":240,"1d":1440}
mins = INTERVAL_MINS.get(INTERVAL, 5)
real_from = snapshot_utc
real_to   = snapshot_utc + timedelta(minutes=mins * (FORECAST_N + 2))

real_df = fetch_candles_range(SYMBOL, SOURCE, INTERVAL, real_from, real_to)
real_candles = df_to_candles(real_df)[:FORECAST_N]
print(f"Real candles fetched: {len(real_candles)}")
for c in real_candles:
    print(f"  {c['datetime']}  O={c['open']} H={c['high']} L={c['low']} C={c['close']}")

In [ ]:
# ── Score: compare forecast vs reality ───────────────────────────────────────
import pandas as pd

fc_candles   = forecast.get("forecast_candles", [])
fc_direction = forecast.get("direction", "")
fc_target    = forecast.get("key_levels", {}).get("target", 0)
fc_sl        = forecast.get("key_levels", {}).get("stop_loss", 0)

# Actual direction over the period
if real_candles:
    entry_close = hist_candles[-1]["close"]
    final_close = real_candles[-1]["close"]
    actual_direction = "bullish" if final_close > entry_close else ("bearish" if final_close < entry_close else "sideways")
    pct_move = (final_close - entry_close) / entry_close * 100
    target_hit = any(c["high"] >= fc_target for c in real_candles) if fc_target else False
    sl_hit     = any(c["low"]  <= fc_sl    for c in real_candles) if fc_sl     else False
else:
    actual_direction = "unknown"
    pct_move = 0
    target_hit = sl_hit = False

direction_hit = fc_direction == actual_direction

print("=" * 50)
print("FORECAST SCORECARD")
print("=" * 50)
print(f"Forecasted direction : {fc_direction.upper()}")
print(f"Actual direction     : {actual_direction.upper()}")
print(f"Direction HIT        : {'✅' if direction_hit else '❌'}")
print(f"Price move           : {pct_move:+.2f}%")
print(f"Target hit ({fc_target}) : {'✅' if target_hit else '❌'}")
print(f"SL hit     ({fc_sl})    : {'🔴 YES' if sl_hit else '✅ NO'}")
print()

# Candle-by-candle comparison table
rows = []
for i, (fc, real) in enumerate(zip(fc_candles, real_candles), 1):
    fc_dir  = "↑" if fc["close"] > fc["open"] else "↓"
    real_dir = "↑" if real["close"] > real["open"] else "↓"
    match = "✅" if fc_dir == real_dir else "❌"
    rows.append({
        "Candle": i,
        "FC Open": round(fc["open"], 2),
        "FC Close": round(fc["close"], 2),
        "FC Dir": fc_dir,
        "Real Open": round(real["open"], 2),
        "Real Close": round(real["close"], 2),
        "Real Dir": real_dir,
        "Match": match,
        "Close Err %": round(abs(fc["close"] - real["close"]) / real["close"] * 100, 3),
    })

df_compare = pd.DataFrame(rows)
display(df_compare)

In [ ]:
# ── Visualise — full candlestick chart ───────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

from backend.agents.kronos_agent import _ema, _rsi

def _plot_candles_full(candles, ax_c, ax_v, title, color="#c9d1d9",
                       show_ema=True, fc_target=None, fc_sl=None):
    """Full candlestick + volume + EMA 9/21 on given axes."""
    n  = len(candles)
    xs = np.arange(n)

    closes  = [c["close"]  for c in candles]
    volumes = [c["volume"] for c in candles] if "volume" in candles[0] else [0]*n

    e9  = _ema(closes, 9)
    e21 = _ema(closes, 21)

    for i, c in enumerate(candles):
        o, h, l, cl = c["open"], c["high"], c["low"], c["close"]
        col = "#3fb950" if cl >= o else "#f85149"
        ax_c.plot([i, i], [l, h], color=col, linewidth=0.9, zorder=2)
        bh = max(abs(cl - o), 0.001 * cl)
        ax_c.add_patch(mpatches.Rectangle((i - 0.35, min(o, cl)), 0.7, bh,
                                           color=col, linewidth=0, zorder=3))

    if show_ema:
        ax_c.plot(xs, [v or np.nan for v in e9],  color="#58a6ff", linewidth=1,   label="EMA 9",  zorder=4)
        ax_c.plot(xs, [v or np.nan for v in e21], color="#d29922", linewidth=1,   label="EMA 21", zorder=4)
        ax_c.legend(loc="upper left", framealpha=0.3, facecolor="#161b22",
                    labelcolor="#c9d1d9", fontsize=8)

    if fc_target:
        ax_c.axhline(fc_target, color="#3fb950", linestyle="--", linewidth=1,
                     label=f"Target {fc_target:.2f}", alpha=0.85)
        ax_c.text(n - 0.3, fc_target, f" T {fc_target:.2f}",
                  color="#3fb950", fontsize=8, va="center")
    if fc_sl:
        ax_c.axhline(fc_sl, color="#f85149", linestyle="--", linewidth=1,
                     label=f"SL {fc_sl:.2f}", alpha=0.85)
        ax_c.text(n - 0.3, fc_sl, f" SL {fc_sl:.2f}",
                  color="#f85149", fontsize=8, va="center")

    ax_c.set_title(title, color=color, fontsize=10, pad=6)
    ax_c.set_xlim(-0.8, n - 0.2)
    ax_c.grid(True, axis="y", alpha=0.3)
    ax_c.set_ylabel("Price", color="#8b949e", fontsize=8)

    # Volume
    vcols = ["#3fb950" if c["close"] >= c["open"] else "#f85149" for c in candles]
    ax_v.bar(xs, volumes, color=vcols, alpha=0.6, width=0.7)
    ax_v.set_ylabel("Vol", color="#8b949e", fontsize=8)
    ax_v.grid(True, axis="y", alpha=0.3)
    for ax in (ax_c, ax_v):
        ax.set_facecolor("#161b22")
        ax.tick_params(colors="#8b949e", labelsize=8)
        for s in ax.spines.values(): s.set_edgecolor("#30363d")
    ax_v.set_xticks(np.arange(n))
    ax_v.set_xticklabels([f"C{i+1}" for i in range(n)], fontsize=8, color="#8b949e")


# ── Historical context (last 30 candles) + Kronos context ─────────────────────
hist_plot = hist_candles[-30:]   # last 30 history candles as context
fc_plot   = fc_candles[:FORECAST_N]
real_plot = real_candles[:FORECAST_N]

fig = plt.figure(figsize=(22, 13), facecolor="#0d1117")
fig.suptitle(
    f"{SYMBOL} {INTERVAL}  —  Forecast vs Reality  ({FORECAST_AT} IST)\n"
    f"Kronos: {kronos_ctx.get('trend','?').upper()}  RSI={kronos_ctx.get('rsi','?')}  "
    f"Bias={kronos_ctx.get('signal',{}).get('bias','?').upper()}  "
    f"Patterns={kronos_ctx.get('patterns',[])}",
    color="#c9d1d9", fontsize=11, y=0.98
)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3,
                        height_ratios=[3, 1, 0.6])

# Col 0: history context
ax_hc = fig.add_subplot(gs[0, 0])
ax_hv = fig.add_subplot(gs[1, 0], sharex=ax_hc)
_plot_candles_full(hist_plot, ax_hc, ax_hv,
                   title=f"History (last {len(hist_plot)} candles)",
                   color="#8b949e")
plt.setp(ax_hc.get_xticklabels(), visible=False)

# Col 1: forecast
ax_fc = fig.add_subplot(gs[0, 1])
ax_fv = fig.add_subplot(gs[1, 1], sharex=ax_fc)
_plot_candles_full(fc_plot, ax_fc, ax_fv,
                   title=f"Forecast  {fc_direction.upper()}  conf={forecast.get('confidence',0):.0%}",
                   color="#58a6ff",
                   fc_target=fc_target, fc_sl=fc_sl)
plt.setp(ax_fc.get_xticklabels(), visible=False)

# Col 2: reality
ax_rc = fig.add_subplot(gs[0, 2])
ax_rv = fig.add_subplot(gs[1, 2], sharex=ax_rc)
_plot_candles_full(real_plot, ax_rc, ax_rv,
                   title=f"Reality  {actual_direction.upper()}  {pct_move:+.2f}%",
                   color="#d29922",
                   fc_target=fc_target, fc_sl=fc_sl)
plt.setp(ax_rc.get_xticklabels(), visible=False)

# Bottom row: scorecard table
ax_score = fig.add_subplot(gs[2, :])
ax_score.axis("off")
ax_score.set_facecolor("#0d1117")

table_data = [[
    fc_direction.upper(),
    actual_direction.upper(),
    "✅" if direction_hit else "❌",
    f"{pct_move:+.2f}%",
    "✅" if target_hit else "❌",
    "🔴" if sl_hit else "✅",
]]
col_labels = ["FC Dir", "Real Dir", "Dir Hit", "Move", "Target Hit", "SL Safe"]
tbl = ax_score.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    bbox=[0.1, 0.0, 0.8, 1.0],
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
for (row, col), cell in tbl.get_celld().items():
    cell.set_facecolor("#21262d" if row == 0 else "#161b22")
    cell.set_edgecolor("#30363d")
    cell.set_text_props(color="#c9d1d9")

out_png = f"forecast_{SYMBOL}_{INTERVAL}_{FORECAST_AT.replace(' ','_').replace(':','')}.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Chart saved → {out_png}")

In [ ]:
# ── Summary JSON ──────────────────────────────────────────────────────────────
summary = {
    "symbol":             SYMBOL,
    "interval":           INTERVAL,
    "snapshot":           FORECAST_AT,
    "forecast_direction": fc_direction,
    "actual_direction":   actual_direction,
    "direction_hit":      direction_hit,
    "price_move_pct":     round(pct_move, 3),
    "target_hit":         target_hit,
    "sl_hit":             sl_hit,
    "kronos_bias":        kronos_ctx.get("signal", {}).get("bias"),
    "kronos_score":       kronos_ctx.get("signal", {}).get("score"),
    "patterns_detected":  kronos_ctx.get("patterns", []),
    "candle_matches":     [r["Match"] for r in rows],
    "avg_close_err_pct":  round(sum(r["Close Err %"] for r in rows) / max(len(rows), 1), 3),
}
print(json.dumps(summary, indent=2))